# graphArt_LM: prompted Qwen2.5-Coder 14B on the hierarchical schema (dev)

No training. Qwen2.5-Coder 14B through Ollama, with two worked examples from the train split,
runs `pipeline/run_hier.py` on the 51 torchvision **dev** programs: per program one tree call,
then one wiring call per block variant of its own tree. Two jobs, one Ollama server per T4:
- the whole answer (tree + wiring), split over both GPUs;
- wiring alone, given the true tree (`--oracle-tree`), on GPU 1.
Scored with `pipeline/score_hier.py` (headline: `hier_right`). The frozen test sets are not
in this notebook. About 1.5-2 h, capped at 2.5.

**Before running**: *GPU T4 x2*, *Internet on* (installs Ollama, downloads the model, ~9 GB).
No inputs. Then *Save Version -> Save & Run All (Commit)*.

**When it finishes**, download `graphart_hier_14b.zip` from the *Output* tab: results,
scores, raw replies, logs.

Generated by `scripts/make_kaggle_notebook.py --hier-14b`. Edit that script, not this file.

In [ ]:
# MODEL        Ollama tag; q4_K_M, ~9 GB, fits one T4 with its context.
# SHOTS        worked examples before each prompt, from the TRAIN split only, fixed by a rule
#              (pipeline/run_hier.py shots(); saved in prompts/hier_shots2.json).
# NUM_CTX      context per call: the longest dev prompt is ~6k tokens with the examples, and
#              the longest tree answer ~9k. If the model does not fit wholly on one GPU at
#              this size (checked in step 3), FALLBACK_NUM_CTX is used instead.
# RUN_ARGS     output caps: tree 10,240 tokens (densenet121's tree is ~9k), wiring 2,048.
# TIME_BUDGET_HOURS  the whole session; jobs still running at the end are stopped, and every
#              program finished so far is kept and scored.
# VALIDATE_ONLY True: only check the code arrived intact (works on a CPU session).
MODEL = "qwen2.5-coder:14b"; SHOTS = 2; NUM_CTX = 20480; FALLBACK_NUM_CTX = 16384; RUN_ARGS = ["--tree-tokens", "10240", "--wire-tokens", "2048"]; TIME_BUDGET_HOURS = 2.5; PACKAGE_RESERVE_MIN = 6; VALIDATE_ONLY = False

## 1. Unpack the code, prompts and samples

In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request
from pathlib import Path

START = time.time()
WORK = Path(os.environ.get("GRAPHART_WORK", "/kaggle/working"))
REPO = WORK / "graphArt_LM"
BUNDLE_SHA256 = "18fa858444bc58258c17ca2d4b3879f22412c22909f4136b59f1ae253f6d63c1"
BUNDLE = (
    "..."
    "AAAEWVo="
)
data = base64.b64decode(BUNDLE)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256, "bundle damaged -- regenerate the notebook"
REPO.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(data), mode="r:xz") as tar:
    tar.extractall(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(sorted(str(p.relative_to(REPO)) for p in REPO.rglob("*") if p.is_file()))
if VALIDATE_ONLY:
    raise SystemExit("bundle intact -- VALIDATE_ONLY is set, so stopping here")
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True)

## 2. Install Ollama, one server per GPU, pull the model

In [ ]:
import zipfile

def up(port) -> bool:
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{port}/api/version", timeout=2).read()
        return True
    except OSError:
        return False

n_gpu = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
print(f"{n_gpu} GPU(s)", flush=True)
if n_gpu < 1:
    raise SystemExit("no GPU -- set the accelerator to GPU T4 x2")
PORTS = [11434, 11435][:max(1, min(2, n_gpu))]
if not shutil.which("ollama"):
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null",
                   shell=True, check=True)
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
LOGS = REPO / "logs"
LOGS.mkdir(exist_ok=True)
for gpu, port in enumerate(PORTS):
    if up(port):
        continue
    env = dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{port}", OLLAMA_KEEP_ALIVE="24h", OLLAMA_NUM_PARALLEL="1",
               OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", CUDA_VISIBLE_DEVICES=str(gpu))
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open(LOGS / f"ollama{gpu}.log", "w"),
                     stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(1)
        if up(port):
            break
    else:
        raise RuntimeError(f"ollama on GPU {gpu} did not start -- see logs/ollama{gpu}.log")
for attempt in (1, 2, 3):
    r = subprocess.run(["ollama", "pull", MODEL], env=dict(os.environ, OLLAMA_HOST=f"127.0.0.1:{PORTS[0]}"))
    if r.returncode == 0:
        break
else:
    raise SystemExit(f"could not pull {MODEL}")
print("servers:", PORTS, flush=True)

## 3. Warm up each server; does the model fit on one GPU?

In [ ]:
def generate(port, prompt, num_ctx, n=16):
    body = json.dumps({"model": MODEL, "prompt": prompt, "stream": False, "think": False,
                       "options": {"temperature": 0, "num_predict": n, "num_ctx": num_ctx}}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{port}/api/generate", data=body,
                                 headers={"Content-Type": "application/json"})
    return json.loads(urllib.request.urlopen(req, timeout=1800).read())

def on_gpu(port):
    ps = json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/api/ps", timeout=10).read())
    m = [x for x in ps.get("models", []) if x.get("name", "").startswith(MODEL.split(":")[0])]
    return bool(m) and m[0].get("size_vram", 0) >= m[0].get("size", 1), m

for port in PORTS:
    t = time.time()
    r = generate(port, "def add(a, b):", NUM_CTX)
    ok, m = on_gpu(port)
    print(f"port {port}: {time.time() - t:.0f}s, reply {r.get('response', '')[:40]!r}, "
          f"all on GPU: {ok}", flush=True)
    if not ok and NUM_CTX != FALLBACK_NUM_CTX:
        print(f"  partly on CPU at num_ctx {NUM_CTX}: using {FALLBACK_NUM_CTX}", flush=True)
        NUM_CTX = FALLBACK_NUM_CTX
        generate(port, "def add(a, b):", NUM_CTX)
        print("  now all on GPU:", on_gpu(port)[0], flush=True)
# speed, on a real-size prompt: tokens per second of generation
r = generate(PORTS[0], "Write a long Python class for a ResNet.", NUM_CTX, n=200)
print(f"generation {r['eval_count'] / (r['eval_duration'] / 1e9):.1f} tok/s, "
      f"prompt {r['prompt_eval_count'] / max(r['prompt_eval_duration'], 1) * 1e9:.0f} tok/s", flush=True)
subprocess.run("nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv", shell=True)

## 4. Run: the whole answer split over the GPUs, wiring alone on the second

In [ ]:
from graphart.hier import to_text
rows = [json.loads(l) for l in open(REPO / "data/generated/hier_dev.jsonl", encoding="utf-8")]
ntags = {r["id"]: sum(1 for v in r["hier"]["variants"].values() if not v["hidden"]) for r in rows}
work = {r["id"]: len(to_text(r["hier"])) + 600 * ntags[r["id"]] for r in rows}   # ~ output to write
# GPU 1 also runs wiring alone; balance the whole-answer programs around it, longest first
load = [0, sum(600 * n for n in ntags.values())] if len(PORTS) > 1 else [0]
part = {}
for rid in sorted(work, key=lambda k: -work[k]):
    g = min(range(len(load)), key=lambda i: load[i])
    part[rid] = g
    load[g] += work[rid]
COMMON = ["--backend", "ollama", "--model", MODEL, "--split", "hier_dev", "--shots", str(SHOTS),
          "--num-ctx", str(NUM_CTX), *RUN_ARGS]
JOBS = [[] for _ in PORTS]
for g in range(len(PORTS)):
    ids = ",".join(r["id"] for r in rows if part[r["id"]] == g)
    JOBS[g].append(("whole", COMMON + ["--ids", ids, "--part", "ab"[g]]))
JOBS[-1].insert(0, ("oracle_tree", COMMON + ["--oracle-tree"]))
print({g: [n for n, _ in j] for g, j in enumerate(JOBS)}, "| est. load", load, flush=True)

def remaining():
    return TIME_BUDGET_HOURS * 3600 - (time.time() - START)

ZIP_NAME = "graphart_hier_14b.zip"

def package():
    with zipfile.ZipFile(WORK / ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
        for root in (REPO / "results", LOGS):
            for p in root.rglob("*") if root.exists() else []:
                if p.is_file():
                    z.write(p, p.relative_to(REPO))

procs = []
for g, jobs in enumerate(JOBS):
    script = [(name, [sys.executable, "-m", "pipeline.run_hier", *args]) for name, args in jobs]
    runner = ("import subprocess, sys\n"
              f"for name, cmd in {script!r}:\n"
              "    print(f'##### {name}', flush=True)\n"
              "    subprocess.run(cmd)\n")
    log = open(LOGS / f"gpu{g}.log", "w")
    env = dict(os.environ, PYTHONUNBUFFERED="1", OLLAMA_URL=f"http://127.0.0.1:{PORTS[g]}/api/generate")
    procs.append((g, log, subprocess.Popen([sys.executable, "-c", runner], stdout=log,
                                           stderr=subprocess.STDOUT, env=env)))
last = time.time()
while any(p.poll() is None for _, _, p in procs):
    if remaining() < PACKAGE_RESERVE_MIN * 60:
        for g, _, p in procs:
            if p.poll() is None:
                p.kill()
                print(f"GPU {g}: stopped by the time budget -- finished programs are kept", flush=True)
        break
    time.sleep(30)
    if time.time() - last > 600:
        last = time.time()
        for g, _, _ in procs:
            done = [l for l in (LOGS / f"gpu{g}.log").read_text(errors="replace").splitlines()
                    if l.strip().startswith("[")]
            print(f"  GPU {g}: {done[-1].strip() if done else 'starting'}   ({remaining() / 3600:.1f} h left)",
                  flush=True)
for g, log, p in procs:
    log.close()
    print(f"\n##### GPU {g} (exit {p.returncode})\n" +
          "\n".join((LOGS / f"gpu{g}.log").read_text(errors="replace").splitlines()[-20:]), flush=True)
package()

## 5. Merge the two halves, score

In [ ]:
res = REPO / "results"
for parts in {tuple(sorted(res.glob(p.name.split("__part")[0] + "__part*.jsonl")))
              for p in res.glob("*__part*.jsonl")}:
    base = parts[0].name.split("__part")[0]
    order = {r["id"]: i for i, r in enumerate(rows)}
    recs = sorted((json.loads(l) for p in parts for l in p.open(encoding="utf-8")), key=lambda r: order[r["id"]])
    (res / f"{base}.jsonl").write_text("".join(json.dumps(r) + "\n" for r in recs), encoding="utf-8")
    metas = [json.loads(p.with_name(p.stem + ".meta.json").read_text()) for p in parts]
    meta = dict(metas[0], n=len(recs), of=len(rows), failed=sum(m["failed"] for m in metas),
                calls=sum(m["calls"] for m in metas), parts=[p.name for p in parts])
    (res / f"{base}.meta.json").write_text(json.dumps(meta, indent=2))
    print(f"merged {len(parts)} parts -> {base}.jsonl ({len(recs)} programs)", flush=True)
runs = sorted(str(p.relative_to(REPO)) for p in res.glob("ollama__*__hier*.jsonl") if "__part" not in p.name)
out = subprocess.run([sys.executable, "-m", "pipeline.score_hier", *runs], capture_output=True, text=True)
print(out.stdout[-20000:], out.stderr[-3000:], flush=True)
(LOGS / "scores.txt").write_text(out.stdout)
package()
print(f"done in {(time.time() - START) / 3600:.1f} h -- download {ZIP_NAME}")